# Generative Adversarial Networks (GANs)
## Teaching Two Neural Networks to Outsmart Each Other


---

### Learning Objectives
1. Understand the minimax game formulation and Nash equilibrium.
2. Implement DCGAN Generator and Discriminator in PyTorch.
3. Code the alternating training loop with proper gradient blocking.
4. Diagnose and understand mode collapse.
5. Compare GAN variants (WGAN, StyleGAN2, Pix2Pix, CycleGAN) by FID and capability.

### Accessibility Note
All figures use colourblind-safe deep indigo / electric lime palette. Bar charts have hatch patterns. Scatter plots use distinct marker shapes. Alt-text is printed below every figure cell. H1→H2 heading hierarchy supports screen readers.

## Imports and Setup

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.patches import FancyBboxPatch
from scipy import stats
import warnings; warnings.filterwarnings('ignore')

#Deep Indigo / Electric Lime
INDIGO  = '#2D1B69'
LIME    = '#A8E63D'
VIOLET  = '#6B46C1'
LVIOLET = '#B794F4'
XINDIGO = '#EDE9FE'
XLIME   = '#F0FBD8'
CREAM   = '#FAFAF8'
DARK    = '#0D0A1A'
MID     = '#4A4560'
LGREY   = '#E2E0EA'
PINK    = '#EC4899'
CYAN    = '#06B6D4'
RED     = '#EF4444'

plt.rcParams.update({
    'font.family':'DejaVu Sans','figure.facecolor':CREAM,
    'axes.facecolor':CREAM,'axes.spines.top':False,'axes.spines.right':False,
    'axes.linewidth':0.8,
})
np.random.seed(42)
print('Setup complete. Theme: Deep Indigo / Electric Lime')

---
## 1 The Minimax Game

The GAN objective is a **two-player minimax game**:

$$\min_G \max_D V(G,D) = \mathbb{E}[\log D(x)] + \mathbb{E}[\log(1 - D(G(z)))]$$

- **D maximises** V: it wants to correctly classify real (→1) and fake (→0)
- **G minimises** V: it wants D to classify its fakes as real (→1)
- **Nash equilibrium**: D*(x) = 0.5 everywhere G perfectly matches Pdata

### Non-Saturating Loss
In practice G trains on `max log D(G(z))` rather than `min log(1 - D(G(z)))`.  
This avoids vanishing gradients when D is winning early in training.

In [ ]:
# Demonstrate the value function terms
def discriminator_value(d_real, d_fake):
    """Compute V(G,D) terms. D wants both terms high; G wants 2nd term low."""
    term1 = np.log(d_real + 1e-8)            # E[log D(x)]
    term2 = np.log(1 - d_fake + 1e-8)        # E[log(1 - D(G(z)))]
    return term1, term2

print('V(G,D) at different training stages:')
print(f'{'Stage':<25} {'E[log D(x)]':>14} {'E[log(1-D(G))]:':>16} {'V total':>10}')
print('-'*70)
for stage, dr, df in [
    ('Early (D naive)',       0.55, 0.45),
    ('D winning',            0.95, 0.05),
    ('G catching up',        0.80, 0.30),
    ('Nash equilibrium',     0.50, 0.50),
]:
    t1, t2 = discriminator_value(dr, df)
    print(f'{stage:<25} {t1:>14.3f} {t2:>16.3f} {t1+t2:>10.3f}')

In [ ]:
# GAN Architecture Diagram
fig,ax=plt.subplots(figsize=(13,5.2),facecolor=CREAM)
ax.set_facecolor(CREAM); ax.set_xlim(0,14); ax.set_ylim(0,7); ax.axis('off')
fig.suptitle('Figure 1  GAN Architecture: Generator (Forger) vs Discriminator (Detective)',
             fontsize=12.5,fontweight='bold',color=DARK,y=0.97)

def box(ax,x,y,w,h,fc,ec,txt,fs=9.5,bold=False):
    r=FancyBboxPatch((x,y),w,h,boxstyle='round,pad=0.12',facecolor=fc,edgecolor=ec,lw=2.2,zorder=3)
    ax.add_patch(r)
    ax.text(x+w/2,y+h/2,txt,ha='center',va='center',fontsize=fs,
            color=DARK,fontweight='bold' if bold else 'normal',linespacing=1.45,zorder=4)

def arr(ax,x1,y1,x2,y2,color=DARK,lw=2.0):
    ax.annotate('',xy=(x2,y2),xytext=(x1,y1),
                arrowprops=dict(arrowstyle='->',color=color,lw=lw,mutation_scale=15))

box(ax,0.2,2.8,1.4,1.4,'#1A1035',LIME,'z~N(0,I)\nLatent\nNoise',8.5)
box(ax,2.1,2.3,2.0,2.2,XINDIGO,INDIGO,'Generator\nG(z)\n"The Forger"',10,True)
box(ax,4.7,2.7,1.4,1.4,XLIME,LIME,'FAKE\nIMAGE\nG(z)',9)
box(ax,4.7,4.5,1.4,1.4,'#FFF7ED','#D97706','REAL\nIMAGE\ndata',9)
box(ax,6.7,2.3,2.0,2.2,XINDIGO,VIOLET,'Discriminator\nD(x)\n"The Detective"',10,True)
box(ax,9.2,2.7,1.4,1.4,XINDIGO,INDIGO,'P(real)\n0=fake\n1=real',9)
box(ax,11.2,3.8,2.2,0.85,'#FEF2F2',RED,'D loss:\nclassify',8)
box(ax,11.2,2.6,2.2,0.85,XLIME,LIME,'G loss:\nfool D',8)

arr(ax,1.6,3.5,2.1,3.5,DARK)
arr(ax,4.1,3.4,4.7,3.4,LIME,2.2)
arr(ax,6.1,3.4,6.7,3.4,DARK)
arr(ax,6.1,4.0,6.7,4.0,'#D97706',2.0)
arr(ax,8.7,3.4,9.2,3.4,DARK)
arr(ax,10.6,4.2,11.2,4.2,RED)
arr(ax,10.6,3.0,11.2,3.0,LIME)

ax.annotate('',xy=(3.1,2.3),xytext=(3.1,1.4),arrowprops=dict(arrowstyle='->',color=LIME,lw=2.0))
ax.annotate('',xy=(10.3,1.4),xytext=(3.1,1.4),arrowprops=dict(arrowstyle='<-',color=LIME,lw=2.0))
ax.annotate('',xy=(10.3,1.4),xytext=(10.3,2.6),arrowprops=dict(arrowstyle='<-',color=LIME,lw=2.0))
ax.text(6.7,1.0,'← gradient flows back to G (generator learns from D decisions)',
        ha='center',fontsize=8.5,color=LIME,fontweight='bold',style='italic')

plt.tight_layout(pad=0.5)
plt.savefig('fig1_gan_architecture.png',dpi=160,bbox_inches='tight',facecolor=CREAM)
plt.show()
print('\n[ALT-TEXT] Figure 1: Horizontal flow diagram. Left: dark noise box (z~N(0,I)) with arrow to indigo Generator box labelled The Forger. Lime arrow leads to fake image box. Separate orange real image box feeds into violet Discriminator box (The Detective). Output P(real) box on right. Red D-loss box and lime G-loss box at far right. A lime curved arrow arcs below the whole diagram showing gradient flows back to G. Colourblind-safe indigo/lime/violet/orange palette.')

---
## DCGAN Implementation in PyTorch

DCGAN (Radford et al., 2015) uses three architectural rules that make GAN training reliable:
1. **Strided convolutions** instead of pooling for downsampling
2. **BatchNorm** in both G and D
3. **LeakyReLU(0.2)** in D, **ReLU** in G; **Tanh** as the output activation

The Generator maps latent vector z ∈ ℝ¹⁰⁰ to a 28×28 image via transposed convolutions (upsampling).

In [ ]:
# DCGAN PyTorch implementation (full, runnable)
# Note: Training requires GPU for speed; architecture shown here runs on CPU for demo

PYTORCH_CODE = '''
import torch
import torch.nn as nn

class Generator(nn.Module):
    """DCGAN Generator: noise z -> fake 28x28 MNIST image."""
    def __init__(self, latent_dim=100, img_channels=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 128*7*7),
            nn.ReLU(True),
            nn.Unflatten(1, (128, 7, 7)),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),  # 14x14
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.ConvTranspose2d(64, img_channels, 4, stride=2, padding=1),  # 28x28
            nn.Tanh(),  # output in [-1, 1]
        )
    def forward(self, z): return self.net(z)

class Discriminator(nn.Module):
    """DCGAN Discriminator: image -> P(real)."""
    def __init__(self, img_channels=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(img_channels, 64, 4, stride=2, padding=1),  # 14x14
            nn.LeakyReLU(0.2, inplace=True),  # LeakyReLU, NOT ReLU!
            nn.Conv2d(64, 128, 4, stride=2, padding=1),  # 7x7
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Flatten(),
            nn.Linear(128*7*7, 1),
            nn.Sigmoid(),  # P(real) in [0, 1]
        )
    def forward(self, img): return self.net(img)

# Training loop
criterion = nn.BCELoss()
opt_G = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

for real_imgs, _ in dataloader:
    batch = real_imgs.size(0)
    # Step 1: Train Discriminator
    z = torch.randn(batch, 100).to(device)
    fake = G(z).detach()  # detach: block gradients from reaching G
    d_loss = (criterion(D(real_imgs), torch.ones(batch,1)) +
              criterion(D(fake), torch.zeros(batch,1))) / 2
    opt_D.zero_grad(); d_loss.backward(); opt_D.step()
    # Step 2: Train Generator
    g_loss = criterion(D(G(torch.randn(batch,100))), torch.ones(batch,1))
    opt_G.zero_grad(); g_loss.backward(); opt_G.step()
'''
print(PYTORCH_CODE)

In [ ]:
# Minimax Value Function + Training Dynamics
fig=plt.figure(figsize=(13,4.8),facecolor=CREAM)
gs=GridSpec(1,2,figure=fig,wspace=0.42,left=0.08,right=0.97,top=0.85,bottom=0.15)
fig.suptitle('Figure 2   Minimax Objective and Ideal Training Dynamics',
             fontsize=12.5,fontweight='bold',color=DARK,y=0.97)

ax1=fig.add_subplot(gs[0]); ax1.set_facecolor(CREAM)
x=np.arange(2); w=0.32
ax1.bar(x-w/2,[1.0,1.0],w,color=VIOLET,hatch='//',edgecolor=INDIGO,lw=1.3,alpha=0.88,label='D wants HIGH')
ax1.bar(x+w/2,[0.4,0.0],w,color=XLIME,hatch='xx',edgecolor=LIME,lw=1.3,alpha=0.90,label='G wants LOW')
ax1.set_xticks(x); ax1.set_xticklabels(['E[log D(x)]\n(real images)','E[log(1-D(G(z)))]\n(fake images)'],fontsize=9,color=MID)
ax1.set_ylim(0,1.4); ax1.set_ylabel('Desired value',fontsize=10,color=DARK)
ax1.set_title('A  |  Minimax Value Function',fontsize=10.5,fontweight='bold',color=DARK,loc='left')
ax1.legend(fontsize=8.5,frameon=False)
ax1.text(0.5,1.22,'Nash equilibrium: D(x)=0.5 everywhere',ha='center',fontsize=8.5,color=INDIGO,fontweight='bold',
         bbox=dict(boxstyle='round,pad=0.3',facecolor=XINDIGO,edgecolor=INDIGO,lw=1.2))
ax1.spines['left'].set_color(LGREY); ax1.spines['bottom'].set_color(LGREY)

ax2=fig.add_subplot(gs[1]); ax2.set_facecolor(CREAM)
epochs=np.arange(100)
d_loss=0.693+0.18*np.sin(epochs*0.3)*np.exp(-epochs/40)+np.random.normal(0,0.03,100)
g_loss=np.clip(1.8*np.exp(-epochs/35)+0.693+0.15*np.sin(epochs*0.25+1)*np.exp(-epochs/50)+np.random.normal(0,0.04,100),0.4,2.2)
ax2.plot(epochs,d_loss,color=VIOLET,lw=2.5,label='Discriminator loss')
ax2.plot(epochs,g_loss,color=LIME,lw=2.5,linestyle='--',label='Generator loss')
ax2.axhline(np.log(2),color=LGREY,lw=1.5,linestyle=':')
ax2.text(95,np.log(2)+0.04,'log 2=0.693\n(Nash)',fontsize=7.5,color=MID,ha='right',style='italic')
ax2.fill_between(epochs[60:],d_loss[60:]-0.08,d_loss[60:]+0.08,alpha=0.12,color=VIOLET)
ax2.set_xlabel('Training Iteration',fontsize=10,color=DARK)
ax2.set_ylabel('Loss',fontsize=10,color=DARK)
ax2.set_title('B  |  Both Losses Converge to log 2',fontsize=10.5,fontweight='bold',color=DARK,loc='left')
ax2.legend(fontsize=9,frameon=False)
ax2.set_ylim(0.2,2.3); ax2.set_xlim(0,99)
ax2.spines['left'].set_color(LGREY); ax2.spines['bottom'].set_color(LGREY)

plt.tight_layout()
plt.savefig('fig2_minimax_training.png',dpi=160,bbox_inches='tight',facecolor=CREAM)
plt.show()
print('\n[ALT-TEXT] Figure 2: Two-panel figure. Left (A): grouped bar chart. Violet bars with forward-slash hatch = D wants HIGH. Lime bars with cross hatch = G wants LOW. Indigo annotation box shows Nash equilibrium. Right (B): line chart over 100 iterations. Violet solid = Discriminator loss; lime dashed = Generator loss. Both converge to log(2)=0.693 (grey dotted line). Violet shaded band shows post-convergence range. Colourblind-safe indigo/lime/violet palette, hatch patterns on bars.')

---
## 3 DCGAN Architecture and Latent Space

In [ ]:
# DCGAN Architecture + Latent Space
fig=plt.figure(figsize=(13,5.0),facecolor=CREAM)
gs=GridSpec(1,2,figure=fig,wspace=0.38,left=0.04,right=0.97,top=0.85,bottom=0.08)
fig.suptitle('Figure 3  DCGAN Architecture and Latent Space Structure',
             fontsize=12.5,fontweight='bold',color=DARK,y=0.97)

ax1=fig.add_subplot(gs[0]); ax1.set_facecolor(CREAM)
ax1.set_xlim(0,13); ax1.set_ylim(0,7); ax1.axis('off')
ax1.set_title('A  |  DCGAN Generator: Noise to Image',fontsize=10.5,fontweight='bold',color=DARK,loc='left')

g_blocks=[
    (0.2,2.6,1.3,1.8,'#1A1035',LIME,'z\n(100,)'),
    (1.9,2.3,1.6,2.4,XINDIGO,INDIGO,'Linear\n+Reshape\n128x7x7'),
    (3.9,2.1,1.8,2.8,XINDIGO,VIOLET,'ConvT\n64ch\n14x14\nBN+ReLU'),
    (6.1,2.1,1.8,2.8,XINDIGO,VIOLET,'ConvT\n1ch\n28x28\nTanh'),
    (8.3,2.3,1.6,2.4,XLIME,LIME,'Fake\nImage\n28x28'),
]
centres_g=[(bx+bw/2,by+bh/2) for bx,by,bw,bh,_fc,_ec,_t in g_blocks]
for bx,by,bw,bh,fc,ec,txt in g_blocks:
    r=FancyBboxPatch((bx,by),bw,bh,boxstyle='round,pad=0.10',facecolor=fc,edgecolor=ec,lw=2.0,zorder=3)
    ax1.add_patch(r); ax1.text(bx+bw/2,by+bh/2,txt,ha='center',va='center',fontsize=8.5,color=DARK,fontweight='bold',linespacing=1.4,zorder=4)
for i in range(len(g_blocks)-1):
    x0=g_blocks[i][0]+g_blocks[i][2]; x1=g_blocks[i+1][0]; y0=centres_g[i][1]; y1=centres_g[i+1][1]
    ax1.annotate('',xy=(x1+0.04,y1),xytext=(x0-0.04,y0),arrowprops=dict(arrowstyle='->',color=DARK,lw=1.8,mutation_scale=13))
ax1.text(5.0,0.9,'Strided ConvTranspose  |  BatchNorm  |  ReLU (G) / LeakyReLU (D)',ha='center',fontsize=7.5,color=MID,style='italic')

d_blocks=[
    (1.9,0.1,1.8,0.75,XINDIGO,INDIGO,'Conv 64ch\nLeakyReLU'),
    (4.1,0.1,1.8,0.75,XINDIGO,VIOLET,'Conv 128ch\nBN+LReLU'),
    (6.3,0.1,1.4,0.75,XINDIGO,VIOLET,'Flatten\nLinear'),
    (8.1,0.1,1.2,0.75,'#FEF2F2',RED,'Sigmoid\nP(real)'),
]
ax1.text(0.9,0.48,'Image->',fontsize=8,color=MID,ha='center')
for bx,by,bw,bh,fc,ec,txt in d_blocks:
    r=FancyBboxPatch((bx,by),bw,bh,boxstyle='round,pad=0.08',facecolor=fc,edgecolor=ec,lw=1.5,zorder=3)
    ax1.add_patch(r); ax1.text(bx+bw/2,by+bh/2,txt,ha='center',va='center',fontsize=7.5,color=DARK,fontweight='bold',linespacing=1.3,zorder=4)
ax1.text(5.0,6.8,'Discriminator (D):',fontsize=8,color=MID,style='italic',ha='center')

# Right: Latent space clusters
ax2=fig.add_subplot(gs[1]); ax2.set_facecolor(CREAM)
ax2.set_title('B  |  Latent Space: Digit Clusters and Interpolation',fontsize=10.5,fontweight='bold',color=DARK,loc='left')
digit_centers=[(0,0),(2,1),(0,2),(-2,1),(-2,-1),(0,-2)]
digit_cols=[LIME,CYAN,PINK,VIOLET,'#F59E0B',RED]
digit_mks=['o','s','^','D','v','P']
def clust(cx,cy,n,std=0.35): return np.random.normal([cx,cy],std,(n,2))
for i,(cx,cy) in enumerate(digit_centers):
    pts=clust(cx,cy,35)
    ax2.scatter(pts[:,0],pts[:,1],s=28,marker=digit_mks[i],color=digit_cols[i],edgecolors=DARK,linewidths=0.4,alpha=0.75,zorder=3,label=f'Digit {i}')
z0=np.array([-2.0,-1.0]); z1=np.array([2.0,1.0])
path=np.linspace(0,1,8)[:,None]*z1+(1-np.linspace(0,1,8)[:,None])*z0
ax2.plot(path[:,0],path[:,1],color='white',lw=1.5,linestyle='--',zorder=5,alpha=0.7)
ax2.scatter(path[:,0],path[:,1],s=25,color='white',marker='o',zorder=6,alpha=0.8)
ax2.scatter([z0[0]],[z0[1]],s=100,color=LIME,marker='*',zorder=7,edgecolors=DARK,lw=1)
ax2.scatter([z1[0]],[z1[1]],s=100,color=CYAN,marker='*',zorder=7,edgecolors=DARK,lw=1)
ax2.text(z0[0]+0.1,z0[1]+0.25,'z0',fontsize=9,color=LIME,fontweight='bold')
ax2.text(z1[0]+0.1,z1[1]+0.25,'z1',fontsize=9,color=CYAN,fontweight='bold')
ax2.set_xlabel('Latent dim 1',fontsize=10,color=DARK)
ax2.set_ylabel('Latent dim 2',fontsize=10,color=DARK)
ax2.legend(fontsize=7.5,frameon=False,ncol=3,loc='lower right',labelspacing=0.3)
ax2.spines['left'].set_color(LGREY); ax2.spines['bottom'].set_color(LGREY)

plt.tight_layout()
plt.savefig('fig3_dcgan_architecture.png',dpi=160,bbox_inches='tight',facecolor=CREAM)
plt.show()
print('\n[ALT-TEXT] Figure 3: Two-panel figure. Left (A): DCGAN Generator flow diagram with dark noise box (z,100) connecting through indigo and violet ConvTranspose blocks to lime fake image output. Below, smaller Discriminator pipeline shown. Right (B): 2D scatter of latent space projection. Six digit classes (0-5) use distinct marker shapes (circle, square, triangle, diamond, down-triangle, plus) in lime, cyan, pink, violet, amber, red. White dashed interpolation path with circle markers connects two starred points z0 (lime) and z1 (cyan). Colourblind-safe palette.')

---
## Mode Collapse

Mode collapse occurs when G learns to produce **one or a few outputs** that reliably fool D, and stops exploring. Solutions:

| Solution | Mechanism | Effectiveness |
|----------|-----------|---------------|
| Minibatch discrimination | D compares samples within a batch | Moderate |
| Wasserstein GAN (WGAN) | Replaces JS with Wasserstein distance | High |
| Spectral normalisation | Lipschitz constraint on D | High |
| Unrolled GANs | G trains against future D states | Moderate |
| Progressive growing | Train at low res first | Very High |

In [ ]:
# Mode Collapse
fig=plt.figure(figsize=(13,4.8),facecolor=CREAM)
gs=GridSpec(1,2,figure=fig,wspace=0.42,left=0.08,right=0.97,top=0.85,bottom=0.14)
fig.suptitle('Figure 4  Mode Collapse: Failure Mode and Solutions',
             fontsize=12.5,fontweight='bold',color=DARK,y=0.97)

ax1=fig.add_subplot(gs[0]); ax1.set_facecolor(CREAM)
ax1.set_title('A  |  Healthy vs Collapsed Output Distribution',fontsize=10.5,fontweight='bold',color=DARK,loc='left')
def mclust(cx,cy,n,std=0.28): return np.random.normal([cx,cy],std,(n,2))
for (cx,cy),col,mk,lab in [((-1.5,-1),LIME,'o','Mode 0'),((-1,1.2),CYAN,'s','Mode 1'),
                             ((0.2,-1.5),PINK,'^','Mode 2'),((1.5,0.5),VIOLET,'D','Mode 3'),
                             ((0,-0.2),'#F59E0B','v','Mode 4')]:
    pts=mclust(cx,cy,40)
    ax1.scatter(pts[:,0],pts[:,1],s=35,marker=mk,color=col,edgecolors=DARK,linewidths=0.5,alpha=0.75,zorder=4,label=lab)
collapsed=mclust(0.3,0.1,200,std=0.16)
ax1.scatter(collapsed[:,0],collapsed[:,1],s=18,marker='x',color=RED,alpha=0.45,zorder=2,label='Collapsed output')
ax1.add_patch(plt.Circle((0.3,0.1),0.42,fill=False,edgecolor=RED,lw=2.2,linestyle='--',zorder=5))
ax1.text(0.3,-0.55,'MODE\nCOLLAPSE',ha='center',fontsize=8,color=RED,fontweight='bold')
ax1.set_xlabel('Latent dim 1',fontsize=10,color=DARK); ax1.set_ylabel('Latent dim 2',fontsize=10,color=DARK)
ax1.legend(fontsize=7.5,frameon=False,ncol=2,loc='upper right',labelspacing=0.3)
ax1.spines['left'].set_color(LGREY); ax1.spines['bottom'].set_color(LGREY)

ax2=fig.add_subplot(gs[1]); ax2.set_facecolor(CREAM)
ax2.set_title('B  |  Solutions: Effectiveness vs Cost',fontsize=10.5,fontweight='bold',color=DARK,loc='left')
solutions=['Minibatch\nDiscrim.','Wasserstein\nGAN','Spectral\nNorm','Unrolled\nGANs','Progressive\nGrowing']
benefit=[6,9,8,5,9]; complexity=[4,7,5,8,9]
xp=np.arange(5); w=0.33
ax2.bar(xp-w/2,benefit,w,color=LIME,hatch='//',edgecolor=INDIGO,lw=1.2,alpha=0.88,label='Effectiveness (1-10)')
ax2.bar(xp+w/2,complexity,w,color=LVIOLET,hatch='xx',edgecolor=VIOLET,lw=1.2,alpha=0.88,label='Implementation cost (1-10)')
for i,(b,c) in enumerate(zip(benefit,complexity)):
    ax2.text(xp[i]-w/2,b+0.15,str(b),ha='center',fontsize=8,fontweight='bold',color=INDIGO)
    ax2.text(xp[i]+w/2,c+0.15,str(c),ha='center',fontsize=8,fontweight='bold',color=VIOLET)
ax2.set_xticks(xp); ax2.set_xticklabels(solutions,fontsize=8,color=MID)
ax2.set_ylabel('Score (1-10)',fontsize=10,color=DARK); ax2.set_ylim(0,11.5)
ax2.legend(fontsize=8.5,frameon=False)
ax2.spines['left'].set_color(LGREY); ax2.spines['bottom'].set_color(LGREY)

plt.tight_layout()
plt.savefig('fig4_mode_collapse.png',dpi=160,bbox_inches='tight',facecolor=CREAM)
plt.show()
print('\n[ALT-TEXT] Figure 4: Two-panel figure. Left (A): 2D scatter of generator output distribution. Five healthy clusters use distinct markers (circle, square, triangle, diamond, down-triangle) in lime, cyan, pink, violet, amber. Red X markers in a dashed red circle labelled MODE COLLAPSE show a collapsed generator. Right (B): grouped bar chart of five solutions. Lime bars with forward-slash hatch = effectiveness; light violet bars with cross hatch = implementation cost. All bars labelled with scores. Colourblind-safe indigo/lime/violet palette, hatch patterns on all bars.')

In [ ]:
# GAN Variants
fig=plt.figure(figsize=(13,4.8),facecolor=CREAM)
gs=GridSpec(1,2,figure=fig,wspace=0.42,left=0.08,right=0.97,top=0.85,bottom=0.16)
fig.suptitle('Figure 5  GAN Variants: Stability vs Capability and FID Progress',
             fontsize=12.5,fontweight='bold',color=DARK,y=0.97)

ax1=fig.add_subplot(gs[0]); ax1.set_facecolor(CREAM)
variants=['DCGAN\n(2015)','WGAN\n(2017)','Pix2Pix\n(2017)','CycleGAN\n(2017)','StyleGAN\n(2019)','StyleGAN2\n(2020)']
stability=[5,9,7,7,8,9]; capability=[5,6,8,9,9,10]
sizes=[80,100,100,100,180,200]; mkrs=['o','s','^','D','v','P']
cols=[LIME,CYAN,PINK,VIOLET,'#F59E0B',INDIGO]
for lab,st,ca,sz,mk,col in zip(variants,stability,capability,sizes,mkrs,cols):
    ax1.scatter(st,ca,s=sz,marker=mk,color=col,edgecolors=DARK,linewidths=1.3,zorder=4,label=lab)
ax1.set_xlabel('Training Stability (1-10)',fontsize=10,color=DARK)
ax1.set_ylabel('Capability / Versatility (1-10)',fontsize=10,color=DARK)
ax1.set_title('A  |  GAN Variants: Stability vs Capability',fontsize=10.5,fontweight='bold',color=DARK,loc='left')
ax1.legend(fontsize=7.8,frameon=False,ncol=2,loc='lower right',labelspacing=0.3)
ax1.set_xlim(3.5,10.5); ax1.set_ylim(3.5,11.0)
ax1.grid(True,alpha=0.15,color=LGREY)
ax1.spines['left'].set_color(LGREY); ax1.spines['bottom'].set_color(LGREY)

ax2=fig.add_subplot(gs[1]); ax2.set_facecolor(CREAM)
years=['DCGAN\n2015','WGAN\n2017','ProGAN\n2018','StyleGAN\n2019','StyleGAN2\n2020']
fid_vals=[80,50,8.8,4.4,2.8]
hatch_b=['//', 'xx', '\\\\\\\\', 'xx', '..']
col_b=[LIME,CYAN,PINK,VIOLET,INDIGO]
bars=ax2.bar(years,fid_vals,color=col_b,hatch=hatch_b,edgecolor='white',linewidth=1.5,alpha=0.90,width=0.62)
for bar,val in zip(bars,fid_vals):
    ax2.text(bar.get_x()+bar.get_width()/2,val+0.8,f'{val}',ha='center',fontsize=9,fontweight='bold',color=DARK)
ax2.set_ylabel('FID Score  (lower = better)',fontsize=10,color=DARK)
ax2.set_title('B  |  FID Progress 2015-2020',fontsize=10.5,fontweight='bold',color=DARK,loc='left')
ax2.set_ylim(0,92)
ax2.annotate('',xy=(4.3,3.5),xytext=(0.0,82),arrowprops=dict(arrowstyle='->',color=LIME,lw=2.0,connectionstyle='arc3,rad=-0.3'))
ax2.text(2.2,68,'Rapid improvement\n2015-2020',fontsize=8,color=INDIGO,ha='center',fontweight='bold')
ax2.spines['left'].set_color(LGREY); ax2.spines['bottom'].set_color(LGREY)

plt.tight_layout()
plt.savefig('fig5_gan_variants.png',dpi=160,bbox_inches='tight',facecolor=CREAM)
plt.show()
print('\n[ALT-TEXT] Figure 5: Two-panel comparison. Left (A): scatter of training stability vs capability for six GAN variants. Distinct markers: circle DCGAN, square WGAN, triangle Pix2Pix, diamond CycleGAN, down-triangle StyleGAN, plus StyleGAN2. Colourblind-safe lime, cyan, pink, violet, amber, indigo colours. Right (B): bar chart of FID scores from DCGAN (80) to StyleGAN2 (2.8). Each bar has distinct hatch and colour. Lime curved arrow shows rapid improvement trend. All bars labelled. Colourblind-safe palette, hatch patterns on all bars.')

---
## Summary

| Concept | Key Point |
|---------|----------|
| GAN objective | Minimax: G minimises, D maximises V(G,D) |
| Nash equilibrium | D*(x) = 0.5 everywhere; G matches Pdata |
| DCGAN rules | Strided conv, BatchNorm, LeakyReLU |
| Training loop | Alternate: freeze G when training D, vice versa |
| Mode collapse | G collapses to 1 mode; use WGAN or spectral norm |
| FID progress | 80 (DCGAN 2015) → 2.8 (StyleGAN2 2020) |

## References

1. Goodfellow, I. et al. (2014) 'Generative adversarial nets', NeurIPS 27. https://arxiv.org/abs/1406.2661
2. Radford, A., Metz, L. and Chintala, S. (2015) 'DCGAN', arXiv:1511.06434. https://arxiv.org/abs/1511.06434
3. Arjovsky, M., Chintala, S. and Bottou, L. (2017) 'Wasserstein GAN', ICML 2017. https://arxiv.org/abs/1701.07875
4. Karras, T., Laine, S. and Aila, T. (2019) 'StyleGAN', CVPR 2019. https://arxiv.org/abs/1812.04948
5. Isola, P. et al. (2017) 'Pix2Pix', CVPR 2017. https://arxiv.org/abs/1611.07004
6. Zhu, J.Y. et al. (2017) 'CycleGAN', ICCV 2017. https://arxiv.org/abs/1703.10593

---
**Licence:** MIT see LICENSE file